<a href="https://colab.research.google.com/github/chewanna7-code/NatureInsightStudy/blob/main/Summary_and_Extraction_of_NI_info.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# HYDROLOGICAL SCENARIO EXTRACTION
# ============================================================

import pandas as pd
import io
from google.colab import files
from openpyxl import load_workbook

# ============================================================
# BLOCK 1: PARSE FILENAME
# ============================================================

def parse_hydro_filename(filename):
    name = filename.replace(".xlsx", "").replace(".XLSX", "").upper()
    parts = name.split("-")

    result = {
        "Filename": filename,
        "SS": None,
        "Rank": None,
        "RP (years)": None,
        "Catchment": None,
        "Scenario Type": None,
    }

    for part in parts:
        if part.startswith("SS") and part[2:].isdigit():
            result["SS"] = int(part[2:])
        elif part.startswith("R") and part[1:].isdigit():
            result["Rank"] = int(part[1:])
        elif part.endswith("Y") and part[:-1].isdigit():
            result["RP (years)"] = int(part[:-1])
        elif part in ["CAL", "WAN"]:
            result["Catchment"] = part
        elif part in ["ALL", "OBS", "OBH", "OBC", "LS"]:
            result["Scenario Type"] = part

    if result["Scenario Type"] is None:
        result["Scenario Type"] = "Baseline"

    return result

# ============================================================
# BLOCK 2: EXTRACT PEAK FLOW AND TIME TO PEAK
# ============================================================

def extract_hydro_metrics(file_bytes, filename):
    wb = load_workbook(io.BytesIO(file_bytes), data_only=True)
    ws = wb.active

    total_time_col = None
    total_flow_col = None
    header_row = None

    # Scan header row for Total Time and Total Flow columns
    for row in ws.iter_rows():
        for cell in row:
            val = str(cell.value).strip().lower() if cell.value else ""
            if "total" in val and "time" in val and "sec" in val:
                total_time_col = cell.column
                header_row = cell.row
            if "total" in val and "flow" in val and "time" not in val:
                total_flow_col = cell.column
        if total_time_col and total_flow_col:
            break

    if not total_time_col or not total_flow_col:
        print(f"  WARNING: Could not find columns in {filename}")
        print(f"  total_time_col={total_time_col}, total_flow_col={total_flow_col}")
        return {
            "Peak Flow (m³/s)": None,
            "Time to Peak (sec)": None,
            "Time to Peak (hrs)": None,
        }

    # Extract time series data
    times = []
    flows = []

    for row_idx in range(header_row + 1, ws.max_row + 1):
        t = ws.cell(row_idx, total_time_col).value
        f = ws.cell(row_idx, total_flow_col).value
        if t is not None and f is not None:
            try:
                times.append(float(t))
                flows.append(float(f))
            except:
                pass

    if not flows:
        print(f"  WARNING: No flow data found in {filename}")
        return {
            "Peak Flow (m³/s)": None,
            "Time to Peak (sec)": None,
            "Time to Peak (hrs)": None,
        }

    peak_flow = max(flows)
    peak_idx = flows.index(peak_flow)
    time_to_peak_sec = times[peak_idx]
    time_to_peak_hrs = round(time_to_peak_sec / 3600, 4)

    return {
        "Peak Flow (m³/s)": round(peak_flow, 4),
        "Time to Peak (sec)": time_to_peak_sec,
        "Time to Peak (hrs)": time_to_peak_hrs,
    }

# ============================================================
# BLOCK 3: UPLOAD AND PROCESS
# ============================================================

print("Upload all your hydrological Excel files now...")
uploaded = files.upload()

all_rows = []

for filename, file_bytes in uploaded.items():
    print(f"\nProcessing: {filename}")
    meta = parse_hydro_filename(filename)
    metrics = extract_hydro_metrics(file_bytes, filename)
    row = {**meta, **metrics}
    all_rows.append(row)
    print(f"  Catchment={meta['Catchment']}, Scenario={meta['Scenario Type']}, RP={meta['RP (years)']}yr")
    print(f"  Peak Flow={metrics['Peak Flow (m³/s)']} m³/s, Time to Peak={metrics['Time to Peak (hrs)']} hrs")

# ============================================================
# BLOCK 4: BUILD SUMMARY TABLE WITH % REDUCTIONS
# ============================================================

df = pd.DataFrame(all_rows)
df = df.sort_values(by=["Catchment", "Scenario Type", "RP (years)"]).reset_index(drop=True)

# Get baseline values per catchment and RP
baselines = df[df["Scenario Type"] == "Baseline"][
    ["Catchment", "RP (years)", "Peak Flow (m³/s)", "Time to Peak (sec)"]
].rename(columns={
    "Peak Flow (m³/s)": "Baseline Peak Flow",
    "Time to Peak (sec)": "Baseline TTP"
})

df = df.merge(baselines, on=["Catchment", "RP (years)"], how="left")

df["% Peak Reduction"] = df.apply(
    lambda r: round((r["Baseline Peak Flow"] - r["Peak Flow (m³/s)"]) / r["Baseline Peak Flow"] * 100, 2)
    if pd.notna(r["Baseline Peak Flow"]) and pd.notna(r["Peak Flow (m³/s)"]) and r["Scenario Type"] != "Baseline"
    else None, axis=1
)

df["% Slower (TTP)"] = df.apply(
    lambda r: round((r["Time to Peak (sec)"] - r["Baseline TTP"]) / r["Baseline TTP"] * 100, 2)
    if pd.notna(r["Baseline TTP"]) and pd.notna(r["Time to Peak (sec)"]) and r["Scenario Type"] != "Baseline"
    else None, axis=1
)

df = df.drop(columns=["Baseline Peak Flow", "Baseline TTP"])

print("\n\nFULL SUMMARY TABLE:")
display(df)

# ============================================================
# BLOCK 5: PIVOT TABLES
# ============================================================

print("\nPEAK FLOW PIVOT:")
pivot_pf = df.pivot_table(
    index="RP (years)",
    columns=["Catchment", "Scenario Type"],
    values="Peak Flow (m³/s)"
)
display(pivot_pf)

print("\n% PEAK REDUCTION PIVOT:")
pivot_red = df.pivot_table(
    index="RP (years)",
    columns=["Catchment", "Scenario Type"],
    values="% Peak Reduction"
)
display(pivot_red)

# ============================================================
# BLOCK 6: EXPORT TO EXCEL
# ============================================================

output_file = "Hydro_Summary.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Full Summary", index=False)
    pivot_pf.to_excel(writer, sheet_name="Peak Flow Pivot")
    pivot_red.to_excel(writer, sheet_name="% Reduction Pivot")

files.download(output_file)
print(f"\nDone! Saved: {output_file}")

Upload all your hydrological Excel files now...


Saving SS70-R1-100Y-OBH-CAL.xlsx to SS70-R1-100Y-OBH-CAL.xlsx
Saving SS70-R1-10Y-OBH-CAL.xlsx to SS70-R1-10Y-OBH-CAL.xlsx
Saving SS70-R1-100Y-LS-CAL.xlsx to SS70-R1-100Y-LS-CAL.xlsx
Saving SS70-R1-10Y-LS-CAL.xlsx to SS70-R1-10Y-LS-CAL.xlsx
Saving SS70-R1-500Y-OBS-CAL.xlsx to SS70-R1-500Y-OBS-CAL.xlsx
Saving SS70-R1-200Y-OBS-CAL.xlsx to SS70-R1-200Y-OBS-CAL.xlsx
Saving SS70-R1-100Y-OBS-CAL.xlsx to SS70-R1-100Y-OBS-CAL.xlsx
Saving SS70-R1-50Y-OBS-CAL.xlsx to SS70-R1-50Y-OBS-CAL.xlsx
Saving SS70-R1-20Y-OBS-CAL.xlsx to SS70-R1-20Y-OBS-CAL.xlsx
Saving SS70-R1-10Y-OBS-CAL.xlsx to SS70-R1-10Y-OBS-CAL.xlsx
Saving SS70-R1-500Y-ALL-CAL.xlsx to SS70-R1-500Y-ALL-CAL.xlsx
Saving SS70-R1-200Y-ALL-CAL.xlsx to SS70-R1-200Y-ALL-CAL.xlsx
Saving SS70-R1-100Y-ALL-CAL.xlsx to SS70-R1-100Y-ALL-CAL.xlsx
Saving SS70-R1-50Y-ALL-CAL.xlsx to SS70-R1-50Y-ALL-CAL.xlsx
Saving SS70-R1-20Y-ALL-CAL.xlsx to SS70-R1-20Y-ALL-CAL.xlsx
Saving SS70-R1-10Y-ALL-CAL.xlsx to SS70-R1-10Y-ALL-CAL.xlsx
Saving SS70-R1-500Y-CAL.xlsx

,Filename,SS,Rank,RP (years),Catchment,Scenario Type,Peak Flow (m³/s),Time to Peak (sec),Time to Peak (hrs),% Peak Reduction,% Slower (TTP)
0,SS70-R1-10Y-ALL-CAL.xlsx,70,1,10,CAL,ALL,241.9227,109800.0,30.5000,7.06,2.23
1,SS70-R1-20Y-ALL-CAL.xlsx,70,1,20,CAL,ALL,281.2261,109200.0,30.3333,6.82,1.68
2,SS70-R1-50Y-ALL-CAL.xlsx,70,1,50,CAL,ALL,333.1080,109200.0,30.3333,7.68,1.68
3,SS70-R1-100Y-ALL-CAL.xlsx,70,1,100,CAL,ALL,391.8021,112800.0,31.3333,4.29,5.03
4,SS70-R1-200Y-ALL-CAL.xlsx,70,1,200,CAL,ALL,461.7249,107400.0,29.8333,0.00,0.00
5,SS70-R1-500Y-ALL-CAL.xlsx,70,1,500,CAL,ALL,537.4646,107400.0,29.8333,0.00,0.00
6,SS70-R1-10Y-CAL.xlsx,70,1,10,CAL,Baseline,260.3059,107400.0,29.8333,NaN,NaN
7,SS70-R1-20Y-CAL.xlsx,70,1,20,CAL,Baseline,301.8062,107400.0,29.8333,NaN,NaN
8,SS70-R1-50Y-CAL.xlsx,70,1,50,CAL,Baseline,360.8379,107400.0,29.8333,NaN,NaN
9,SS70-R1-100Y-CAL.xlsx,70,1,100,CAL,Baseline,409.3707,107400.0,29.8333,NaN,NaN



PEAK FLOW PIVOT:


Catchment           CAL                                        
Scenario Type       ALL  Baseline        LS       OBH       OBS
RP (years)                                                     
10             241.9227  260.3059  255.4256  256.1335  233.6978
20             281.2261  301.8062       NaN       NaN  267.6353
50             333.1080  360.8379       NaN       NaN  314.3975
100            391.8021  409.3707  409.3786  409.3640  351.8667
200            461.7249  461.7249       NaN       NaN  410.1331
500            537.4646  537.4646       NaN       NaN  527.5853


% PEAK REDUCTION PIVOT:


Catchment       CAL                  
Scenario Type   ALL    LS  OBH    OBS
RP (years)                           
10             7.06  1.87  1.6  10.22
20             6.82   NaN  NaN  11.32
50             7.68   NaN  NaN  12.87
100            4.29  0.00  0.0  14.05
200            0.00   NaN  NaN  11.17
500            0.00   NaN  NaN   1.84

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Saved: Hydro_Summary.xlsx
